In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

9e508f2212
Course: When does the course start?
A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.


In [4]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("GEMINI_API_KEY")
endpoint = "https://generativelanguage.googleapis.com/v1beta/openai/"
model='gemini-2.5-flash'

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "9e508f2212", "course": "data-engineering-zoomcamp", "section": "General Course-Related Questions", "question": "Course: When does the course start?", "answer": "A new cohort runs roughly January\\u2013April every year. For the current cohort\'s exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\\n\\n- Register via the link in the course repo before the cohort starts.\\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\\n- Join DataTalks.Club\'s [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}'

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.beta.chat.completions.parse(
    model=model,
    messages=messages,
    response_format=Questions
)

In [18]:
result  = response.choices[0].message.parsed
print(result.questions)

['When does the Data Engineering Zoomcamp usually take place each year?', 'Where can I find the exact start date and the registration link for the upcoming cohort?', 'What steps should I follow to register for the course?', 'How will I receive official announcements and updates for the course?', 'Is there a community forum or chat where I can connect with other students and ask questions?']


In [19]:
response.usage

CompletionUsage(completion_tokens=80, prompt_tokens=275, total_tokens=864, completion_tokens_details=None, prompt_tokens_details=None)

In [20]:
response.usage.completion_tokens

80

In [21]:
response.usage.prompt_tokens

275

In [22]:
from evaluation_utils import llm_structured

In [28]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['When does the Data Engineering Zoomcamp usually take place during the year?', 'Where can I find the exact start date and registration details for the current cohort?', 'What steps do I need to follow to register for the course?', 'Is there a specific channel to join for important course announcements?', 'How can I connect with other students and get help during the course?']


In [24]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 5.775e-05,
 'output_cost': 0.0012375,
 'total_cost': 0.0012952500000000002}

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'When does the Data Engineering Zoomcamp usually take place each year?',
  'document': '9e508f2212'},
 {'question': 'Where can I find the exact start date for the current cohort and the registration link?',
  'document': '9e508f2212'},
 {'question': "What's the process for registering for the course?",
  'document': '9e508f2212'},
 {'question': 'How can I receive important announcements about the Data Engineering course?',
  'document': '9e508f2212'},
 {'question': 'Is there a community channel or chat for students to connect and ask questions?',
  'document': '9e508f2212'}]